# Evaluation (development period, 2016-2023)

Runs the tests fixed in `docs/preregistration.md`.
The holdout (2024-2026) is not touched here; it lives in `04_holdout.ipynb` and is run once at the end.

Sign convention: Thesis A (informed shorting) predicts a **negative** IC, meaning high short volume leads to underperformance.

In [ ]:
import numpy as np
import pandas as pd

from shortvol import evaluate as ev, features, finra, prices

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
V0 = features.estimate_v0(features.add_all(panel, days))["v0"]  # 2016-2019 short volume only
df = features.add_all(panel, days, v0=V0)

SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume"]
TARGETS = ["fwd_oc", "fwd_cc"]
for s in SIGNALS:
    df[s + "_n"] = ev.neutralize(df, s, CONTROLS)
df["size_tercile"] = df.groupby("date").log_adv20.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["small", "mid", "large"]))
df["offex_tercile"] = df.groupby("date").off_exchange_share.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["low", "mid", "high"]))

In [ ]:
d = df[df.date <= "2023-12-29"]
print(f"development stock-days: {len(d):,}, days: {d.date.nunique()}")

## 1. Headline: 4 signals x 2 targets, raw and neutralized
Raw IC is the signal as is.
Neutralized IC removes day-t return, 5-day return, size (20-day dollar volume), abnormal volume and industry.
The gap between them shows how much of the signal is reversal, size or liquidity in disguise.

In [ ]:
rows = []
for s in SIGNALS:
    for t in TARGETS:
        for kind, col in [("raw", s), ("neutralized", s + "_n")]:
            rows.append({"signal": s, "target": t, "version": kind, **ev.summarize_ic(ev.daily_ic(d, col, t))})
headline = pd.DataFrame(rows).set_index(["signal", "target", "version"])
headline

## 2. Quintile long-short (primary target)
Long the top quintile of the signal, short the bottom quintile, equal weight, rebalanced daily.
Turnover is the daily sum of absolute weight changes (a full flip of both legs is 4).
Break-even is the one-way cost in bps that erases the gross spread. It is reported, not pass/fail.

In [ ]:
pd.DataFrame({f"{s}_n": ev.quintile_spread(d, s + "_n", "fwd_oc") for s in SIGNALS}).T

## 3. Decay
IC of the primary neutralized signal against the open-to-close return 1, 2, 5, 10 and 20 days ahead.
Informed trading should decay slowly; price pressure and reversal should die within a day or two.

In [ ]:
pd.DataFrame({k: ev.summarize_ic(ev.daily_ic(d.assign(y=ev.shift_target(d, "fwd_oc", k)), "svr5_n", "y"))
              for k in [1, 2, 5, 10, 20]}).T

## 4. Splits (primary signal, primary target)
By size, by period, and by off-exchange share.
Thesis B predicts the relation weakens or flips where off-exchange share is high.

In [ ]:
d = d.assign(period=pd.cut(d.date, pd.to_datetime(["2015-12-31", "2019-12-31", "2021-12-31", "2023-12-31"]),
                           labels=["2016-19", "2020-21", "2022-23"]))
pd.concat({
    "size": ev.ic_by_group(d, "svr5_n", "fwd_oc", "size_tercile"),
    "period": ev.ic_by_group(d, "svr5_n", "fwd_oc", "period"),
    "off-exchange share": ev.ic_by_group(d, "svr5_n", "fwd_oc", "offex_tercile"),
})

## 5. Success bar (development part)
1. Mean IC negative with |NW t| >= 3.
2. The same sign in at least 2 of the 3 development periods.
Criterion 3 (holdout) is checked in `04_holdout.ipynb`.

In [ ]:
primary = headline.loc[("svr5", "fwd_oc", "neutralized")]
periods = ev.ic_by_group(d, "svr5_n", "fwd_oc", "period")
c1 = primary.mean_ic < 0 and abs(primary.nw_t) >= 3
c2 = (np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum() >= 2
print(f"1. mean IC {primary.mean_ic:.4f}, NW t {primary.nw_t:.2f}: {'PASS' if c1 else 'FAIL'}")
print(f"2. same sign in {(np.sign(periods.mean_ic) == np.sign(primary.mean_ic)).sum()} of 3 periods: {'PASS' if c2 else 'FAIL'}")